In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def beam_splitter(r):
    t = np.sqrt(1 - r**2)
    U = np.array([[r, t], [t, -r]])
    return np.kron(U, np.eye(2))

dev = qml.device('default.qubit')

@qml.qnode(dev)
def mz_interferometer(r):
    qml.Hadamard(wires=0)
    qml.RX(np.pi, wires=1)
    qml.QubitUnitary(beam_splitter(r), wires=[0,1])
    m = qml.measure(0)
    qml.cond(m, qml.PauliX)(wires=1)
    qml.QubitUnitary(beam_splitter(r), wires=[0,1])
    qml.RX(np.pi, wires=0)
    return qml.probs(wires=[1,0])

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    probs = mz_interferometer(ins)
    p_a = probs[0,1]
    p_d = probs[1,1]
    p_ac = p_a + probs[1,0]
    outs = [float(p_ac), float(p_d)]
    return json.dumps(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, rtol = 1e-4), "Not the correct probabilities"

test_cases = [
    ('0.5', '[0.8125, 0.1875]'),
    ('0.577350269', '[0.777778, 0.222222]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except Exception as exc:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '0.5'...
Runtime Error. Input unitary must be of shape (4, 4) or (batch_size, 4, 4) to act on 2 wires.
Running test case 1 with input '0.577350269'...
Runtime Error. Input unitary must be of shape (4, 4) or (batch_size, 4, 4) to act on 2 wires.
